# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rana4682/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Build the Feature Vector

I created a feature vector using observed search performance signals that are available before making a prediction.

Selected features:

- CTR
- Average Position
- Search Volume
- Trend Percentage
- Word Count

Missing numerical values are filled with zero. These features are available at the decision moment and are safe for modeling.

In [2]:
import os
import subprocess
import sys
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Rana4682/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Baseline Rule Score
df["baseline_score"] = (
    (1 - df["ctr"].fillna(0)) * 4 +
    (df["avg_position"].fillna(0) / 50) * 3 +
    (df["trend_pct"].abs() / 100) * 3
)

# Reason Code
df["reason_code"] = "HIGH_PRIORITY_REFRESH"

# Action Label
df["action"] = "Refresh"

# Ranking
df = df.sort_values("baseline_score", ascending=False)

# Save Output
os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(df[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].head(20))

                 content_id  baseline_score            reason_code   action
24695  content_dd882c4152ac        1355.530  HIGH_PRIORITY_REFRESH  Refresh
15405  content_a023517539fe         846.327  HIGH_PRIORITY_REFRESH  Refresh
14549  content_d020d42e7fcc         788.807  HIGH_PRIORITY_REFRESH  Refresh
3561   content_22f4d2f58c42         646.786  HIGH_PRIORITY_REFRESH  Refresh
19697  content_4f1966b37335         516.078  HIGH_PRIORITY_REFRESH  Refresh
24726  content_aac5bd559d85         480.412  HIGH_PRIORITY_REFRESH  Refresh
9097   content_32ff84795595         452.062  HIGH_PRIORITY_REFRESH  Refresh
27305  content_ceedad7ba6dc         349.986  HIGH_PRIORITY_REFRESH  Refresh
19525  content_f6e1f66b051e         345.910  HIGH_PRIORITY_REFRESH  Refresh
1132   content_a6c4ef450727         338.719  HIGH_PRIORITY_REFRESH  Refresh
5381   content_f511aa26fadf         305.509  HIGH_PRIORITY_REFRESH  Refresh
10195  content_f560a011a094         256.438  HIGH_PRIORITY_REFRESH  Refresh
25826  conte

In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

features = [
    "ctr",
    "avg_position",
    "search_volume",
    "trend_pct",
    "word_count"
]

feature_df = df[features].copy()

feature_df = feature_df.fillna(0)

print(feature_df.head())

print("\nFeature Shape:", feature_df.shape)

    ctr  avg_position  search_volume  trend_pct  word_count
0  0.76          10.6           10.0      -41.4      3221.0
1  0.05          20.3           90.0      -57.7      2481.0
2  0.09          36.5            0.0      -60.9      3515.0
3  0.49           6.2           10.0      -13.8         0.0
4  0.13          44.0            0.0      -34.7      2803.0

Feature Shape: (30000, 5)


## Feature Notes

| Feature | Meaning | Missing Values | Available When? |
|---------|---------|---------------|----------------|
| CTR | Click-through rate | Filled with 0 | Before prediction |
| Average Position | Average Google ranking | Filled with 0 | Before prediction |
| Search Volume | Estimated monthly search demand | Filled with 0 | Before prediction |
| Trend Percentage | Recent traffic trend | Filled with 0 | Before prediction |
| Word Count | Content length | Filled with 0 | Before prediction |

All selected features are observed before making a recommendation.

In [4]:
print("Missing Values")

display(feature_df.isnull().sum())

Missing Values


,0
ctr,0
avg_position,0
search_volume,0
trend_pct,0
word_count,0


## Leakage Hunt

I checked the selected features for leakage.

I did not use:

- Future-window information
- Product flags
- Label-derived columns

The selected features are observed before the decision point and therefore suitable for modeling.

In [5]:
excluded_columns = [
    "content_id",
    "client_id"
]

print("Leakage Check")

print("Excluded identifier columns:")
print(excluded_columns)

print("\nNo future-window features were used.")
print("No label-derived columns were used.")
print("No product flags were used.")

Leakage Check
Excluded identifier columns:
['content_id', 'client_id']

No future-window features were used.
No label-derived columns were used.
No product flags were used.


## What I Excluded and Why
## Excluded Fields

| Field | Reason |
|-------|--------|
| content_id | Identifier only |
| client_id | Identifier only |
| impression_tier | Derived category |
| position_tier | Derived category |
| trend_direction | Derived from trend percentage |

These fields were excluded because they are identifiers or derived values that do not provide independent predictive information.

In [6]:
print("Excluded Fields")

excluded = pd.DataFrame({
    "Field":[
        "content_id",
        "client_id",
        "impression_tier",
        "position_tier",
        "trend_direction"
    ],
    "Reason":[
        "Identifier",
        "Identifier",
        "Derived category",
        "Derived category",
        "Derived from trend percentage"
    ]
})

display(excluded)

Excluded Fields


,Field,Reason
0,content_id,Identifier
1,client_id,Identifier
2,impression_tier,Derived category
3,position_tier,Derived category
4,trend_direction,Derived from trend percentage


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.